# 🧪 LocateAnything-3B · Đếm SẢN PHẨM (bản CHUẨN, tự chứa)

Dùng **đúng recipe detector đã chạy được** (như notebook Kaggle của bạn): nạp
`device_map="auto"` + `apply_chat_template` + `generate` thường + `max_new_tokens=1024`.
Áp dụng cho 3 video của bạn (2 kiện hàng + 1 cà chua), vạch NGANG.

Chạy lần lượt các cell. **Cell 8 (DEBUG)** in raw output của model + vẽ box trên 1 frame
→ thấy NGAY model có bắt được vật không. Nếu Colab yêu cầu *Restart runtime* sau cell 1
thì Restart rồi chạy tiếp từ cell 2.

In [ ]:
# Cell 1 — Cài thư viện (đúng bản notebook Kaggle dùng được)
!pip uninstall -y opencv-python opencv-contrib-python opencv-python-headless -q
!pip install -q -U "transformers==4.57.1" "opencv-python-headless==4.11.0.86" \
    "Pillow==11.1.0" "decord==0.6.0" "lmdb==1.7.5" accelerate peft "supervision>=0.21" matplotlib
print("✅ Đã cài. Nếu Colab báo 'Restart runtime' → Restart rồi chạy tiếp từ Cell 2.")

In [ ]:
# Cell 2 — Imports + kiểm tra GPU
import os, re, time, glob, shutil, subprocess, urllib.request
from dataclasses import dataclass
from typing import List, Tuple, Optional
import numpy as np, cv2, torch, supervision as sv, transformers
from PIL import Image
import matplotlib.pyplot as plt

print("transformers:", transformers.__version__, "(cần 4.57.1)")
print("torch       :", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print("GPU         :", p.name, f"{p.total_memory/1024**3:.1f}GB")

In [ ]:
# Cell 3 — parse_boxes: đổi text model → bbox (nguyên từ notebook chạy được)
NORM_SCALE = 1000
_RE_BOX = re.compile(r"<box><(\d+)><(\d+)><(\d+)><(\d+)></box>")
_RE_BRK = re.compile(r"\[\s*([\d.]+)\s*,\s*([\d.]+)\s*,\s*([\d.]+)\s*,\s*([\d.]+)\s*\]")
_RE_LOC = re.compile(r"<loc_(\d+)>")

@dataclass
class Detection:
    bbox: Tuple[int, int, int, int]
    class_name: str
    confidence: float

def _add(dets, x1, y1, x2, y2, w, h, cls, conf, scale):
    x1, x2 = int(x1/scale*w), int(x2/scale*w)
    y1, y2 = int(y1/scale*h), int(y2/scale*h)
    if x2 > x1 and y2 > y1:
        dets.append(Detection((x1, y1, x2, y2), cls, conf))

def parse_boxes(text, class_name, w, h, default_conf=0.85):
    dets = []
    for m in _RE_BOX.findall(text):
        x1, y1, x2, y2 = (int(v) for v in m)
        _add(dets, x1, y1, x2, y2, w, h, class_name, default_conf, NORM_SCALE)
    if not dets:
        for m in _RE_BRK.findall(text):
            v = [float(x) for x in m]
            sc = NORM_SCALE if max(v) > 1.5 else 1
            _add(dets, v[0], v[1], v[2], v[3], w, h, class_name, default_conf, sc)
    if not dets:
        locs = _RE_LOC.findall(text)
        for i in range(0, len(locs)-3, 4):
            x1, y1, x2, y2 = (int(v) for v in locs[i:i+4])
            _add(dets, x1, y1, x2, y2, w, h, class_name, default_conf, NORM_SCALE)
    return dets

In [ ]:
# Cell 4 — LocateAnythingDetector (NGUYÊN recipe chạy được: device_map auto + generate thường)
class LocateAnythingDetector:
    def __init__(self, model_dir, max_new_tokens=1024):
        self.model_dir = model_dir
        self.max_new_tokens = max_new_tokens
        self._loaded = False
        self.dtype = torch.float16          # T4 (Turing) không có bfloat16 kernel

    def load(self):
        from transformers import AutoTokenizer, AutoProcessor, AutoConfig, AutoModel
        t0 = time.time()
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_dir, trust_remote_code=True)
        self.processor = AutoProcessor.from_pretrained(self.model_dir, trust_remote_code=True)
        config = AutoConfig.from_pretrained(self.model_dir, trust_remote_code=True)
        self.model = AutoModel.from_pretrained(
            self.model_dir, config=config, trust_remote_code=True,
            torch_dtype=self.dtype, device_map="auto", attn_implementation="sdpa")
        self.model.eval()
        self._loaded = True
        print(f"✅ Loaded in {time.time()-t0:.1f}s")
        if torch.cuda.is_available():
            print(f"   GPU Mem: {torch.cuda.memory_allocated()/1024**3:.1f} GB")
        return self

    def _prep_input(self, v):
        if isinstance(v, np.ndarray):
            v = torch.from_numpy(v)
        if torch.is_tensor(v):
            if v.is_floating_point():
                return v.to(device=self.model.device, dtype=torch.float16)
            return v.to(self.model.device)
        return v

    def detect_pil(self, pil_image, prompt, max_new_tokens=None):
        if not self._loaded: self.load()
        w, h = pil_image.size
        max_tok = max_new_tokens or self.max_new_tokens
        messages = [{"role": "user", "content": [
            {"type": "image"}, {"type": "text", "text": f"Locate all instances of: {prompt}"}]}]
        text_prompt = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.processor(text=text_prompt, images=[pil_image], return_tensors="pt")
        inputs = {k: self._prep_input(v) for k, v in inputs.items()}
        with torch.no_grad():
            output = self.model.generate(**inputs, max_new_tokens=max_tok,
                                         do_sample=False, use_cache=True, tokenizer=self.tokenizer)
        if isinstance(output, (list, tuple)) and hasattr(output[0], "shape"):
            raw = self.tokenizer.decode(output[0], skip_special_tokens=True)
        elif hasattr(output, "shape"):
            raw = self.tokenizer.decode(output[0] if output.dim() > 1 else output, skip_special_tokens=True)
        else:
            raw = str(output)
        return parse_boxes(raw, prompt, w, h), raw

    def detect_frame(self, bgr_frame, prompt, max_new_tokens=None):
        pil = Image.fromarray(cv2.cvtColor(bgr_frame, cv2.COLOR_BGR2RGB))
        return self.detect_pil(pil, prompt, max_new_tokens)

In [ ]:
# Cell 5 — Lấy 3 video của bạn (từ repo) + cấu hình
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else ("/content" if os.path.isdir("/content") else os.getcwd())
REPO = os.path.join(WORK, "VisionOS")
BRANCH = "claude/locate-anything-test-suite-xwju2f"
if not os.path.isdir(os.path.join(REPO, ".git")):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    "https://github.com/nguyendinhhuyht20032004-ai/VisionOS.git", REPO], check=True)
VID = os.path.join(REPO, "VisionOS", "sample_videos")

VIDEOS = [
  {"name": "Kiện hàng con lăn", "file": "packages_rollers.mp4", "prompt": "object", "orient": "horizontal", "line_pos": 0.65},
  {"name": "Kiện hàng có nhãn", "file": "packages_belt.mp4",    "prompt": "object", "orient": "horizontal", "line_pos": 0.60},
  {"name": "Cà chua",           "file": "tomatoes_sorting.mp4", "prompt": "tomato", "orient": "horizontal", "line_pos": 0.72},
]
for v in VIDEOS:
    v["path"] = os.path.join(VID, v["file"])
    print(("OK  " if os.path.exists(v["path"]) else "MISSING  "), v["path"])

RESOLUTION = (1280, 720)   # đưa mọi frame về cỡ này (như notebook chạy được)
MAX_FRAMES = 40            # số frame mỗi video (LA chậm → để nhỏ)
STRIDE     = 1             # =1 để track liên tục
MAX_NEW_TOKENS = 1024      # QUAN TRỌNG: 1024 như notebook (đừng giảm — sẽ cụt box)

In [ ]:
# Cell 6 — Tải model + patch bfloat16->float16 cho T4 (nguyên notebook chạy được)
from huggingface_hub import snapshot_download
MODEL_ID = "nvidia/LocateAnything-3B"
print("📥 Downloading model... (lần đầu ~6GB)")
model_dir = snapshot_download(MODEL_ID)
mc = os.path.expanduser("~/.cache/huggingface/modules/transformers_modules")
if os.path.exists(mc): shutil.rmtree(mc)
f = os.path.join(model_dir, "modeling_locateanything.py")
if os.path.exists(f):
    real_f = os.path.realpath(f); code_txt = open(real_f).read()
    old = "pixel_values = pixel_values.to(self.language_model.dtype)"
    if old in code_txt:
        open(real_f, "w").write(code_txt.replace(old, "pixel_values = pixel_values.to(torch.float16)  # T4 fix"))
        print("✅ Patched bfloat16 -> float16")
print(f"📁 {model_dir}\n✅ Ready!")

In [ ]:
# Cell 7 — Nạp mô hình MỘT LẦN
detector = LocateAnythingDetector(model_dir=model_dir, max_new_tokens=MAX_NEW_TOKENS)
detector.load()

In [ ]:
# Cell 8 — 🔍 DEBUG: detect 1 frame/video → IN raw output + số box + VẼ box
# Nhìn cell này để biết model CÓ bắt được vật không TRƯỚC khi chạy đếm.
fig, axes = plt.subplots(1, len(VIDEOS), figsize=(19, 5))
for ax, v in zip(np.atleast_1d(axes), VIDEOS):
    cap = cv2.VideoCapture(v["path"]); n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.set(cv2.CAP_PROP_POS_FRAMES, max(0, n // 2)); ok, fr = cap.read(); cap.release()
    fr = cv2.resize(fr, RESOLUTION)
    t0 = time.time()
    dets, raw = detector.detect_frame(fr, v["prompt"])
    print(f"\n=== {v['name']} | prompt='{v['prompt']}' → {len(dets)} box ({time.time()-t0:.1f}s) ===")
    print("RAW (300 ký tự đầu):", repr(raw[:300]))
    img = fr.copy()
    for d in dets:
        x1, y1, x2, y2 = d.bbox
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{v['name']}: {len(dets)} box"); ax.axis("off")
plt.tight_layout(); plt.show()
print("\n→ 0 box? Xem RAW ở trên: nếu model in toạ độ theo định dạng lạ, gửi tôi RAW để chỉnh parse.")
print("  Nếu ra 1 box to = cả khung → đổi prompt (object/box/tomato) ở Cell 5.")

In [ ]:
# Cell 9 — ĐẾM cắt vạch + gán nhãn supervision + xuất video + xem inline
from IPython.display import Video, display

def _linezone(orient, pos, w, h):
    if orient == "horizontal":
        y = int(pos * h); return sv.LineZone(start=sv.Point(0, y), end=sv.Point(w, y))
    x = int(pos * w); return sv.LineZone(start=sv.Point(x, 0), end=sv.Point(x, h))

def _to_sv(dets, w, h):
    # bỏ box phủ ~cả khung (không phải vật)
    dets = [d for d in dets if (d.bbox[2]-d.bbox[0])*(d.bbox[3]-d.bbox[1]) <= 0.9*w*h]
    if not dets: return sv.Detections.empty()
    return sv.Detections(
        xyxy=np.array([d.bbox for d in dets], np.float32),
        confidence=np.array([d.confidence for d in dets], np.float32),
        class_id=np.zeros(len(dets), int))

def run_one(v, max_frames=MAX_FRAMES, stride=STRIDE):
    w, h = RESOLUTION
    tracker = sv.ByteTrack(frame_rate=30)
    line = _linezone(v["orient"], v["line_pos"], w, h)
    box_ann = sv.RoundBoxAnnotator(thickness=2)
    lab_ann = sv.LabelAnnotator(text_scale=0.5)
    trace_ann = sv.TraceAnnotator(thickness=2, trace_length=20)
    line_ann = sv.LineZoneAnnotator(thickness=2, text_scale=0.8)
    out = os.path.join(WORK, v["file"].replace(".mp4", "_out.mp4"))
    cap = cv2.VideoCapture(v["path"]); writer = None
    i = used = total_det = 0
    while used < max_frames:
        ok, fr = cap.read()
        if not ok: break
        if i % stride != 0: i += 1; continue
        i += 1; used += 1
        fr = cv2.resize(fr, (w, h))
        dets, _ = detector.detect_frame(fr, v["prompt"])
        sd = _to_sv(dets, w, h); total_det += len(sd)
        sd = tracker.update_with_detections(sd)
        line.trigger(sd)
        an = fr.copy()
        if len(sd):
            an = trace_ann.annotate(an, sd)
            an = box_ann.annotate(an, sd)
            tids = sd.tracker_id if sd.tracker_id is not None else [None]*len(sd)
            labels = [f"{v['prompt']} #{int(t)}" if t is not None else v["prompt"] for t in tids]
            an = lab_ann.annotate(an, sd, labels=labels)
        an = line_ann.annotate(an, line)
        if writer is None:
            writer = cv2.VideoWriter(out, cv2.VideoWriter_fourcc(*"mp4v"), 10, (w, h))
        writer.write(an)
    cap.release()
    if writer: writer.release()
    return dict(name=v["name"], IN=int(line.in_count), OUT=int(line.out_count),
                total=int(line.in_count + line.out_count),
                det=round(total_det / max(used, 1), 1), frames=used, out=out)

print(f"{'video':22}{'IN':>4}{'OUT':>5}{'tổng':>6}{'det/fr':>8}")
print("-" * 45)
rows = []
for v in VIDEOS:
    r = run_one(v); rows.append(r)
    print(f"{r['name'][:21]:22}{r['IN']:>4}{r['OUT']:>5}{r['total']:>6}{r['det']:>8}")

for r in rows:                       # H.264 để xem inline
    h264 = r["out"].replace(".mp4", "_h264.mp4")
    os.system(f"ffmpeg -y -loglevel error -i {r['out']} -vcodec libx264 -pix_fmt yuv420p {h264}")
    print("\n🎬", r["name"])
    display(Video(h264 if os.path.exists(h264) else r["out"], embed=True, width=680))

### Nếu Cell 8 (DEBUG) ra **0 box** hoặc **1 box cả khung**
- Xem dòng **RAW** in ra: đó là text thô model trả về.
  - RAW có `<box>..</box>` hoặc `[x,y,x,y]` → parse OK, chỉ cần đổi prompt.
  - RAW rỗng / lạ → gửi tôi đúng chuỗi RAW đó, tôi chỉnh `parse_boxes`.
- Đổi **prompt** ở Cell 5: thử `object` · `box` · `carton box` · `tomato` · `fruit`.
- LA **chậm** (~vài giây/frame) là bình thường.
- Nếu OOM khi nạp: Restart runtime, chạy lại từ Cell 1 (mỗi cell 1 lần).